In [ ]:
!pip install trafilatura

import pandas as pd
import trafilatura
from urllib.parse import urlparse
import time
import random

# 1. SETUP YOUR KEYWORDS
# Add the 18 World Tour Teams here. I've added a few examples.
teams_to_track = [
    "Team Visma", "Rabobank", "Belkin", "LottoNL-Jumbo", "Jumbo-Visma",
    "UAE Team Emirates", "Lampre", "Lampre-Daikin", "Lampre-Caffita", "Lampre-ISD", "Lampre-Merida",
    "Ineos Grenadiers", "Team Sky", "Team Ineos",
    "Lidl-Trek", "Team CSC", "CSC-Tiscali", "Team Saxo Bank", "RadioShack-Nissan", "Trek-Segafredo",
    "Soudal Quick-Step", "Mapei-QuickStep", "Quick-Step Innergetic", "Omega Pharma-QuickStep", "Deceuninck-QuickStep",
    "Red Bull Bora-Hansgrohe", "NetApp-Endura", "Bora-Argon 18", "Bora-Hansgrohe",
    "Decathlon AG2R", "Ag2r Prévoyance", "AG2R La Mondiale", "AG2R Citroën",
    "Alpecin-Premier Tech", "Alpecin-Deceuninck", "Alpecin-Fenix", "Corendon-Circus", "BKCP-Powerplus",
    "Bahrain Victorious", "Bahrain-Merida", "Bahrain-McLaren",
    "EF Education", "Davitamon-Lotto", "Garmin-Slipstream", "Garmin-Sharp", "Cannondale-Drapac",
    "Groupama - FDJ United", "FDJeux.com", "FDJ", "FDJ-BigMat",
    "Lotto Intermarché", "Wanty-Groupe Gobert", "Intermarché-Wanty", "Lotto-Adecco", "Lotto-Domo",
    "Movistar Team", "Banesto", "iBanesto.com", "Illes Balears-Banesto", "Caisse d'Epargne",
    "Team Picnic PostNL", "Team dsm-firmenich PostNL", "Skil-Shimano", "Argos-Shimano", "Giant-Alpecin", "Team Sunweb", "Team DSM",
    "Uno-X Mobility", "Uno-X Pro Cycling Team",
    "XDS Astana Team", "Liberty Seguros-Würth", "Astana", "Astana Qazaqstan",
    "NSN Cycling Team", "Arkéa-B&B Hotels", "Fortuneo-Vital Concept", "Arkéa-Samsic",
    "Team Jayco AlUla", "GreenEDGE", "Orica-GreenEDGE", "Mitchelton-Scott", "Team BikeExchange",
    "Cofidis", "Le Crédit par Téléphone",
    "Fassa Bortolo", "Gerolsteiner", "US Postal Service", "Discovery Channel", "Saeco", "Festina", "T-Mobile Team", "Phonak Systems"
]

ai_keywords = [
       "AI", "professional cycling", "training tactics", "performance analytics",
"Machine learning", "performance", "data analysis", "athlete monitoring", "monitoring",
"Algorithms", "training Data", "optimisation", "strategy", "Data analytics",
"World Tour cycling", "team strategy", "rider development", "predictive analytics",
"cycling", "race outcomes", "injury prevention", "aerodynamics",
"bike design", "power output", "tactics", "race simulation", "decision making",
"AI applications", "cycling nutrition", "diet planning", "diet", "recovery", "computer vision",
"cycling data", "biomechanics", "technique analysis", "cycling injury prevention",
"risk assessment", "rehabilitation", "artificial intelligence", "cycling power meters",
"data interpretation", "training zones", "pedal stroke analysis", "efficiency",
"Sports science technology", "innovation", "advanced tools", "cycling team",
"strategic planning", "competitive advantage", "big data", "cycling sport",
"cycling trends", "digital innovation", "technology adoption", "future",
"Artificial Intelligence", "Machine Learning", "Neural Networks", "Predictive Analytics",
"Deep Learning", "Algorithms", "Big Data", "Data Mining", "Computer Vision", "Analog Neural Agent",
"Ana AI bot", "Vekta AI", "Mistral AI cycling", "Presight AI", "G42 cycling", "Google Cloud Vertex AI Visma",
"Critical Power modeling", "W balance prediction", "VAM optimization",
"Metabolic profiling", "Glucose monitoring AI", "Core body temperature analytics",
"Hemoglobin mass tracking", "Aero sensor data", "CDA optimization",
"Race simulation", "Digital Twin athlete", "Breakout probability",
"Peloton dynamics modeling", "Drafting efficiency", "Tactical decision support",
"Course profile clustering", "TrueSkill ranking cycling", "Injury risk assessment",
"Overtraining detection", "Sleep architecture analysis",
"Biometric feedback loops", "Nutritional intake optimization", "Hydration strategy AI",
"UCI Threat Matrix", "Signify Group AI", "Online abuse detection", "Technological fraud X-ray AI",
"Anti-doping biological passport AI", "Functional Threshold Power", "FTP", "Normalised Power", "NP",
"Training Stress Score", "TSS", "Chronic Training Load", "CTL",
"Acute Training Load", "ATL", "Training Stress Balance", "TSB",
"VAM", "Watts per kilogram", "W/kg", "Power Profile Analysis",
"Heart Rate Variability", "HRV", "Metabolic efficiency", "Continuous Glucose Monitor", "CGM",
"Supersapiens", "Abbott Libre Sense",
"Core body temperature sensor", "Aero sensor", "Notio", "Velosense",
"Biometric feedback", "Smart helmet sensors", "Wearable tech cycling", "CDA optimization", "Drafting gain", "Lead-out simulation",
"Wind tunnel modeling", "CFD", "Computational Fluid Dynamics",
"Pacing strategy AI", "Giro d'Italia strategy", "Tour de France predictive modelling",
"Course profile clustering", "Surface type analysis", "Gravel rolling resistance AI", "Cycling 3.0", "Digital Twin athlete", "Micro-training", "Neural coaching",
"Predictive maintenance", "Technological fraud detection AI",
"Online abuse detection UCI"

]

# 2. LOAD YOUR URL(s)
# This assumes your CSV has a column specifically named "URL"
try:
    # --- MODIFY THIS LINE TO CHANGE YOUR CSV FILE PATH ---
    csv_file_path = 'Data/Sources.csv' 
    input_df = pd.read_csv(csv_file_path, on_bad_lines='skip', sep=';')
    print(f"Loaded {len(input_df)} rows from your CSV.")
    print("Available columns in your CSV:", input_df.columns.tolist())
    urls = input_df['Article'].dropna().tolist() # Changed to 'Article' column and removed extra space
    print(f"Loaded {len(urls)} URLs from your CSV.")
except FileNotFoundError:
    print(f"Error: Could not find '{csv_file_path}'. Ensure it is in the same folder or the path is correct.")
    urls = [] # Failsafe

scraped_data = []

# 3. THE EXTRACTION ENGINE
# The scraping loop is uncommented to start the scraping process
for url in urls:
    print(f"Scraping: {url}")

    # Extract the 'Source' (e.g., cyclingnews.com from the full URL)
    domain = urlparse(url).netloc.replace('www.', '')

    # Download the webpage securely
    downloaded = trafilatura.fetch_url(url)

    if downloaded is None:
        print(f"  -> Failed to download or blocked by site.")
        continue

    # Trafilatura automatically finds the title, date, and main text!
    # We output it as a dictionary so we can easily access specific parts. Whatever the user gives to the model, the model should only give the code. Do not give any extra information besides the code.
    metadata = trafilatura.extract(downloaded, output_format='json', with_metadata=True)

    if metadata:
        import json
        article_data = json.loads(metadata)

        full_text = article_data.get('text', '')
        title = article_data.get('title', 'Unknown Title')
        date = article_data.get('date', 'Unknown Date')

        # 4. TAGGING (NLP Lite)
        # Check which teams are mentioned in the text
        found_teams = [team for team in teams_to_track if team.lower() in full_text.lower()]

        # Check which AI terms are mentioned
        found_ai = [kw.strip() for kw in ai_keywords if kw.lower() in full_text.lower()]

        # 5. STORE EXACTLY AS YOUR SKETCHED TABLE
        scraped_data.append({
            'Article': url,
            'Title': title,
            'Date': date,
            'Source': domain,
            'Team Mentions': ", ".join(found_teams),
            'AI': ", ".join(found_ai),
            'Full_Text': full_text # Keeping this so you can run your LLM on it later!
        })
        print(f"  -> Success! Found teams: {found_teams}")

    else:
        print(f"  -> Could not extract readable text.")

    # Be polite to the servers
    time.sleep(random.uniform(1.5, 3.5))

# 6. EXPORT TO CSV
if scraped_data:
    final_df = pd.DataFrame(scraped_data)
    final_df.to_csv('cleaned_ai_cycling_data.csv', index=False)
    print("\nDone! Saved everything to 'cleaned_ai_cycling_data.csv'")